<a href="https://colab.research.google.com/github/micah-shull/AI_Agents/blob/main/851_HITLv2_Nodes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# HITL Orchestrator Nodes

## Overall Architecture

Your node layer acts as a **thin orchestration wrapper** around your business logic.

The structure looks like this:

```
LangGraph Workflow

goal_node
   ↓
planning_node
   ↓
data_loading_node
   ↓
routing_node
   ↓
human_simulation_node
   ↓
audit_feedback_node
   ↓
report_node
```

Each node does only one thing:

| Node                  | Responsibility             |
| --------------------- | -------------------------- |
| goal_node             | Define agent mission       |
| planning_node         | Define workflow steps      |
| data_loading_node     | Load datasets              |
| routing_node          | Apply governance policy    |
| human_simulation_node | Simulate reviewer behavior |
| audit_feedback_node   | Generate logs + signals    |
| report_node           | Produce executive report   |

That is **exactly the right level of granularity**.

---

# Strengths in the Design

## 1️⃣ Thin Node Philosophy

Your nodes contain almost no business logic:

```python
decisions = compute_routing_decisions(...)
```

All heavy logic lives in utilities.

This gives you:

```
testable utilities
clean orchestration
easy reuse
```

Which is exactly what you want if you're building a **reusable HITL module** for future agents.

---

# 2️⃣ Config Injection via Closures

This is a **very nice design choice**:

```python
def make_data_loading_node(config)
```

Instead of using globals, you inject configuration.

Benefits:

```
environment portability
testability
multiple agent configs
```

This also keeps nodes **pure functions of state + config**.

That’s excellent architecture.

---

# 3️⃣ Deterministic Pipeline

Your pipeline is fully deterministic because:

```
deterministic human simulation
deterministic routing
structured datasets
```

Meaning:

```
same inputs → same outputs
```

Which reinforces your core agent philosophy:

> Trustworthy agents produce **consistent results**.

---

# 4️⃣ Clean State Mutation

Each node returns **only the fields it modifies**.

Example:

```python
return {"routing_decisions": decisions}
```

This keeps state mutation predictable.

LangGraph merges updates cleanly.

This is **best practice** for state-driven workflows.

---

# 5️⃣ Proper Separation of Concerns

Your utilities handle:

```
data logic
policy evaluation
human behavior
audit creation
report generation
```

Nodes only coordinate execution.

This structure makes your system easy to extend later.

---

# 6️⃣ Automatic Report Saving

Your report node automatically writes the output:

```python
hitl_v2_report_YYYYMMDD_HHMMSS.md
```

This creates a **persistent record of runs**, which is extremely valuable for:

```
debugging
portfolio demos
governance traceability
```

This aligns well with the **audit-first design** of the system.

---

# How This Fits Into Your Overall Agent Philosophy

Your architecture now models a **complete decision governance stack**:

```
AI recommendation
↓
policy enforcement
↓
human oversight
↓
final decision
↓
audit record
↓
feedback signal
↓
executive report
```

This is exactly the kind of system **enterprise leaders trust**.

Most agents only implement the first step.

---

# Small Improvements I Recommend

Nothing here is broken — these are just refinements.

---

# 1️⃣ Add Processing Time Measurement

Your state already includes:

```
processing_time
```

But it isn’t populated.

You could measure total runtime in the report node.

Example:

```python
import time

start = time.time()
...
processing_time = time.time() - start
```

Then return:

```python
"processing_time": processing_time
```

This lets the report show:

```
Total processing time: 0.42 seconds
```

Executives often care about **system latency**.

---

# 2️⃣ Optional Node-Level Error Guard

Currently if a utility throws an exception the node will crash.

You could add lightweight protection:

Example pattern:

```python
try:
    decisions = compute_routing_decisions(...)
except Exception as e:
    state_errors.append(str(e))
    decisions = []
```

Not required for MVP but useful later.

---

# 3️⃣ Persist Audit + Feedback Logs (Future)

Right now you generate:

```
new_audit_entries
new_feedback_entries
```

But they are not written back to the dataset.

Later you could optionally append to:

```
audit_logs.json
feedback_events.json
```

This would simulate **long-term system memory**.

That would allow analytics like:

```
historical override rate
policy violation trends
reviewer performance
```

---

# 4️⃣ Optional Policy Compliance Node (Future)

As we discussed earlier, you could insert:

```
policy_compliance_node
```

between:

```
human_simulation
↓
audit_feedback
```

This would detect:

```
policy violations
routing errors
governance breaches
```

Your architecture is already **perfectly positioned for this**.

---

# Overall Assessment

This node layer is **very well designed**.

Strengths:

```
thin orchestration
clean separation of logic
deterministic workflow
config injection
reusable utilities
clear state updates
```

It aligns extremely well with the **agent development template** you’ve been building.

This would actually be a **great reference example in your repo** for:

> “How to structure LangGraph orchestrator nodes.”

---

# Big Picture Observation

With this final component complete, your HITL system now demonstrates something extremely valuable:

It shows **how to build AI agents that organizations can trust**.

Your architecture includes:

```
AI decision
policy governance
human oversight
audit logging
feedback learning
executive reporting
```

That combination is **very rare in most AI agent examples today**.


In [ ]:
"""HITL v2 orchestrator nodes. Nodes are thin; business logic in utilities."""

from typing import Any, Dict, List

from agents.hitl_v2.orchestrator.utilities.data_loading import load_all_hitl_data
from agents.hitl_v2.orchestrator.utilities.routing import compute_routing_decisions
from agents.hitl_v2.orchestrator.utilities.human_simulation import run_simulated_reviews
from agents.hitl_v2.orchestrator.utilities.audit_feedback import build_audit_entries, build_feedback_entries
from agents.hitl_v2.orchestrator.utilities.report import build_rollup, build_hitl_report


def goal_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Set fixed goal for HITL governance run."""
    return {
        "goal": {
            "name": "HITL Collaboration Governance",
            "description": "Route tasks by policy, simulate human review, log audit and feedback.",
        }
    }


def planning_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Set fixed plan (steps)."""
    return {
        "plan": [
            {"step": "data_loading", "description": "Load tasks, agent outputs, policy, reviewers, and logs."},
            {"step": "routing", "description": "Evaluate routing policy per task."},
            {"step": "human_simulation", "description": "Simulate human review for human_review/escalate."},
            {"step": "audit_feedback", "description": "Build audit entries and feedback events."},
            {"step": "report", "description": "Generate executive report."},
        ]
    }


def make_data_loading_node(config: Any):
    """Data loading node with config (closure)."""
    def data_loading_node(state: Dict[str, Any]) -> Dict[str, Any]:
        payload, errors = load_all_hitl_data(config)
        state_errors = list(state.get("errors") or [])
        state_errors.extend(errors)
        return {
            "tasks": payload.get("tasks"),
            "agent_outputs": payload.get("agent_outputs"),
            "routing_policy": payload.get("routing_policy"),
            "human_reviewers": payload.get("human_reviewers"),
            "human_reviews": payload.get("human_reviews"),
            "audit_logs": payload.get("audit_logs"),
            "feedback_events": payload.get("feedback_events"),
            "tasks_by_id": payload.get("tasks_by_id"),
            "agent_output_by_task_id": payload.get("agent_output_by_task_id"),
            "reviewers_by_role": payload.get("reviewers_by_role"),
            "tasks_count": payload.get("tasks_count", 0),
            "agent_outputs_count": payload.get("agent_outputs_count", 0),
            "reviewers_count": payload.get("reviewers_count", 0),
            "rules_count": payload.get("rules_count", 0),
            "errors": state_errors,
        }
    return data_loading_node


def routing_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Evaluate routing policy for each task."""
    tasks = state.get("tasks") or []
    agent_output_by_task_id = state.get("agent_output_by_task_id") or {}
    policy = state.get("routing_policy") or {}
    decisions = compute_routing_decisions(tasks, agent_output_by_task_id, policy)
    return {"routing_decisions": decisions}


def human_simulation_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Simulate human reviews for human_review and escalate tasks."""
    decisions = state.get("routing_decisions") or []
    tasks_by_id = state.get("tasks_by_id") or {}
    reviewers_by_role = state.get("reviewers_by_role") or {}
    reviews = run_simulated_reviews(decisions, tasks_by_id, reviewers_by_role)
    return {"simulated_reviews": reviews}


def audit_feedback_node(state: Dict[str, Any]) -> Dict[str, Any]:
    """Build new audit entries and feedback events from this run."""
    decisions = state.get("routing_decisions") or []
    reviews = state.get("simulated_reviews") or []
    tasks_by_id = state.get("tasks_by_id") or {}
    agent_output_by_task_id = state.get("agent_output_by_task_id") or {}
    new_audit = build_audit_entries(decisions, reviews, tasks_by_id, agent_output_by_task_id)
    new_feedback = build_feedback_entries(reviews, decisions)
    rollup = build_rollup(decisions, reviews)
    return {
        "new_audit_entries": new_audit,
        "new_feedback_entries": new_feedback,
        "rollup": rollup,
    }


def make_report_node(config: Any):
    """Report node: build markdown and optionally save to file (closure)."""
    def report_node(state: Dict[str, Any]) -> Dict[str, Any]:
        rollup = state.get("rollup") or {}
        routing_decisions = state.get("routing_decisions") or []
        simulated_reviews = state.get("simulated_reviews") or []
        new_audit_entries = state.get("new_audit_entries") or []
        new_feedback_entries = state.get("new_feedback_entries") or []
        tasks_count = state.get("tasks_count", 0)
        agent_outputs_count = state.get("agent_outputs_count", 0)
        reviewers_count = state.get("reviewers_count", 0)
        rules_count = state.get("rules_count", 0)
        errors = state.get("errors") or []

        report_md = build_hitl_report(
            rollup=rollup,
            routing_decisions=routing_decisions,
            simulated_reviews=simulated_reviews,
            new_audit_entries=new_audit_entries,
            new_feedback_entries=new_feedback_entries,
            tasks_count=tasks_count,
            agent_outputs_count=agent_outputs_count,
            reviewers_count=reviewers_count,
            rules_count=rules_count,
            errors=errors,
        )
        report_file_path = None
        reports_dir = getattr(config, "reports_dir", "agents/output/hitl_v2_reports")
        if reports_dir:
            from pathlib import Path
            root = Path(__file__).resolve().parent.parent.parent.parent.parent
            out_dir = root / reports_dir
            out_dir.mkdir(parents=True, exist_ok=True)
            from datetime import datetime, timezone
            stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
            path = out_dir / f"hitl_v2_report_{stamp}.md"
            path.write_text(report_md, encoding="utf-8")
            report_file_path = str(path)
        return {
            "hitl_report": report_md,
            "report_file_path": report_file_path,
        }
    return report_node
